In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
classification_report,
confusion_matrix
)

In [3]:
stock=pd.read_csv("../data/reliance_m1_dataset.csv",
                  index_col=0,
                  parse_dates=True
                 )
stock.head()

,Close,High,Low,Open,Volume,Return,MA_5,MA_20,Volatility_20,Tomorrow_Close,Target,Lag_1,Lag_2,Lag_3
Date,,,,,,,,,,,,,,
2014-01-29,176.457184,179.810844,176.119719,179.241357,12374165,-0.009765,179.623129,182.191623,0.012204,174.010498,0,0.002016,-0.028237,0.002137
2014-01-30,174.010498,175.877159,173.419920,175.360401,14102043,-0.013866,177.902011,181.664320,0.012120,175.307663,1,-0.009765,0.002016,-0.028237
2014-01-31,175.307663,175.792783,174.253056,174.432334,11038353,0.007455,176.362274,181.315244,0.012116,173.145737,0,-0.013866,-0.009765,0.002016
2014-02-03,173.145737,175.022930,172.839895,174.263619,8227762,-0.012332,175.423679,180.956152,0.012182,173.282822,1,0.007455,-0.013866,-0.009765
2014-02-04,173.282822,174.358520,171.511073,171.795823,12362252,0.000792,174.440781,180.739429,0.011796,172.375839,0,-0.012332,0.007455,-0.013866


# Day 8 - Model Improvement

## Objective

Improve the baseline machine learning model by creating additional technical indicators and comparing multiple classification algorithms.

Expected Outcome:
- Better feature set
- Better model performance
- Selection of the best model

In [4]:
stock['EMA_20'] = stock['Close'].ewm(
    span=20,
    adjust=False
).mean()

In [6]:
stock[['Close','EMA_20']].head(25)

,Close,EMA_20
Date,,
2014-01-29,176.457184,176.457184
2014-01-30,174.010498,176.224166
2014-01-31,175.307663,176.136880
2014-02-03,173.145737,175.852009
2014-02-04,173.282822,175.607325
2014-02-05,172.375839,175.299564
2014-02-06,172.080566,174.992993
2014-02-07,171.785278,174.687496
2014-02-10,173.367188,174.561753


In [7]:
stock['Momentum']=stock['Close']-stock['Close'].shift(5)

In [8]:
stock[['Close','Momentum']].head(15)

,Close,Momentum
Date,,
2014-01-29,176.457184,NaN
2014-01-30,174.010498,NaN
2014-01-31,175.307663,NaN
2014-02-03,173.145737,NaN
2014-02-04,173.282822,NaN
2014-02-05,172.375839,-4.081345
2014-02-06,172.080566,-1.929932
2014-02-07,171.785278,-3.522385
2014-02-10,173.367188,0.221451


In [10]:
stock['Price_Range'] = stock['High'] - stock['Low']
stock[['High','Low','Price_Range']].head()

,High,Low,Price_Range
Date,,,
2014-01-29,179.810844,176.119719,3.691125
2014-01-30,175.877159,173.419920,2.457240
2014-01-31,175.792783,174.253056,1.539728
2014-02-03,175.022930,172.839895,2.183035
2014-02-04,174.358520,171.511073,2.847448


In [11]:
stock.dropna(inplace=True)

In [12]:
stock.isnull().sum()

Close             0
High              0
Low               0
Open              0
Volume            0
Return            0
MA_5              0
MA_20             0
Volatility_20     0
Tomorrow_Close    0
Target            0
Lag_1             0
Lag_2             0
Lag_3             0
EMA_20            0
Momentum          0
Price_Range       0
dtype: int64

In [13]:
features = [

'Return',

'MA_5',

'MA_20',

'EMA_20',

'Volatility_20',

'Lag_1',

'Lag_2',

'Lag_3',

'Momentum',

'Price_Range'

]

X = stock[features]

y = stock['Target']

In [14]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2, shuffle=False)

In [18]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [19]:
lr_accuracy=accuracy_score(y_test, lr_pred)
print("Logistic Regression Accuracy: ", lr_accuracy)

Logistic Regression Accuracy:  0.49906890130353815


In [20]:
print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.36      0.02      0.04       265
           1       0.50      0.97      0.66       272

    accuracy                           0.50       537
   macro avg       0.43      0.49      0.35       537
weighted avg       0.43      0.50      0.35       537



In [21]:
print(confusion_matrix(y_test, lr_pred))

[[  5 260]
 [  9 263]]


In [22]:
rf=RandomForestClassifier(
    random_state=42)
rf.fit(X_train, y_train)

rf_pred=rf.predict(X_test)

In [23]:
rf_accuracy=accuracy_score(y_test, rf_pred)
print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.49162011173184356


In [24]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.49      0.77      0.60       265
           1       0.50      0.22      0.31       272

    accuracy                           0.49       537
   macro avg       0.49      0.50      0.45       537
weighted avg       0.49      0.49      0.45       537



In [25]:
print(confusion_matrix(y_test, rf_pred))

[[204  61]
 [212  60]]


In [27]:
gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(X_train,y_train)
gb_pred = gb.predict(X_test)

In [28]:
gb_accuracy = accuracy_score(y_test, gb_pred)

print("Gradient Boosting Accuracy:", gb_accuracy)

Gradient Boosting Accuracy: 0.4934823091247672


In [29]:
print(classification_report(y_test, gb_pred))

              precision    recall  f1-score   support

           0       0.49      0.74      0.59       265
           1       0.50      0.25      0.33       272

    accuracy                           0.49       537
   macro avg       0.50      0.50      0.46       537
weighted avg       0.50      0.49      0.46       537



In [30]:
print(confusion_matrix(y_test, gb_pred))

[[197  68]
 [204  68]]


In [31]:
comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'Gradient Boosting'
    ],
    'Accuracy': [
        lr_accuracy,
        rf_accuracy,
        gb_accuracy
    ]
})

comparison

,Model,Accuracy
0,Logistic Regression,0.499069
1,Random Forest,0.491620
2,Gradient Boosting,0.493482


In [32]:
from sklearn.metrics import roc_auc_score

In [33]:
lr_prob = lr.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, lr_prob)

print("Logistic Regression ROC-AUC:", lr_auc)

Logistic Regression ROC-AUC: 0.4755965593784684


In [34]:
rf_prob = rf.predict_proba(X_test)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC-AUC:", rf_auc)

Random Forest ROC-AUC: 0.5154273029966704


In [35]:
gb_prob = gb.predict_proba(X_test)[:,1]

gb_auc = roc_auc_score(y_test, gb_prob)

print("Gradient Boosting ROC-AUC:", gb_auc)

Gradient Boosting ROC-AUC: 0.5215871254162042


In [36]:
comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'Gradient Boosting'
    ],
    'Accuracy': [
        lr_accuracy,
        rf_accuracy,
        gb_accuracy
    ],
    'ROC-AUC': [
        lr_auc,
        rf_auc,
        gb_auc
    ]
})

comparison

,Model,Accuracy,ROC-AUC
0,Logistic Regression,0.499069,0.475597
1,Random Forest,0.491620,0.515427
2,Gradient Boosting,0.493482,0.521587


In [37]:
stock.to_csv("../data/reliance_improved_dataset.csv")